# Programación Concurrente en Python (Clase 1)
### Introducción a Threads, I/O y Concurrencia

En esta primera sesión aprenderemos:

- Qué es concurrencia.
- Cuál es la diferencia entre concurrencia y paralelismo.
- Qué es un hilo (thread).
- Cómo usar `threading` en Python.
- Por qué la concurrencia funciona bien para tareas de I/O.
- Ejemplos prácticos.

# Librerías y métodos utilizados en esta clase

En esta sesión usamos varias librerías de Python, descritas a continuación:

1. threading  
   Es una librería estándar de Python que permite crear y manejar hilos (threads).  
   Principales elementos usados:
   - Thread: representa un hilo de ejecución.
   - start(): inicia la ejecución del hilo.
   - join(): hace que el programa principal espere a que un hilo termine.

   En esta ocasión usamos threading para ejecutar funciones en paralelo lógico y demostrar concurrencia con operaciones de I/O.

2. requests  
   Es una librería externa (se instala automáticamente en Colab) que permite hacer solicitudes HTTP de una manera sencilla.

   Método usado:
   - requests.get(url): hace una petición GET a una URL y devuelve la respuesta del servidor.  '

   **Importante:** Esto es una operación de I/O, por lo que funciona muy bien con hilos. La usamos para simular tareas que tardan tiempo, como descargar páginas web.

3. time  
   Es una librería estándar que permite trabajar con mediciones de tiempo y pausas.
   Métodos usados:
   - time.sleep(segundos): pausa la ejecución del programa, simulando espera en I/O.
   - time.time(): devuelve el tiempo actual en segundos; se usa para medir cuánto tarda un bloque de código.

# Concurrencia vs Paralelismo

## Concurrencia:
Es la capacidad de un programa de manejar varias tareas "a la vez", alternando entre ellas.
No significa que se ejecuten simultáneamente, sino que avanzan sin bloquearse.

## Paralelismo:
Varias tareas ejecutándose *realmente al mismo tiempo*, gracias a múltiples núcleos de CPU.

## Ejemplo simple:
**Concurrencia:**
Un cocinero alterna entre cortar verduras y revisar la olla.


**Paralelismo:** Dos cocineros trabajando en dos platos al mismo tiempo.

## CASO 1: Proceso SIN Concurrencia

In [2]:
import time

def tarea(nombre):
    print("Iniciando", nombre)
    time.sleep(2)
    print("Terminando", nombre)

print("Ejecutando secuencialmente...")
inicio = time.time()

tarea("A")
tarea("B")

fin = time.time()
print("Tiempo total:", fin - inicio, "segundos")

Ejecutando secuencialmente...
Iniciando A
Terminando A
Iniciando B
Terminando B
Tiempo total: 4.001008987426758 segundos


# Introducción a los Hilos (THreads)
 * Un **hilo** es una unidad de ejecución dentro de un proceso.
* Un programa puede tener 1 hilo o muchos hilos.
* En Python se crean con threading.Thread.

In [3]:
import threading
import time

def tarea(nombre):
    print("Iniciando", nombre)
    time.sleep(2)
    print("Terminando", nombre)

print("Ejecutando concurrentemente con hilos...")
inicio = time.time()

t1 = threading.Thread(target=tarea, args=("A",))
t2 = threading.Thread(target=tarea, args=("B",))

t1.start()
t2.start()

# join() = esperar a que el hilo termine antes de continuar
t1.join()
t2.join()

fin = time.time()
print("Tiempo total:", fin - inicio, "segundos")


Ejecutando concurrentemente con hilos...
Iniciando A
Iniciando B
Terminando A
Terminando B
Tiempo total: 2.00205659866333 segundos


# I/O (Input/Output)
## ¿Qué es I/O?
Son operaciones donde el programa "espera" para despues continuar con su proceso. Puede espara a:
 - Leer archivos
 - Descargar datos de internet
 - Comunicarse con base de datos
 - time.sleep()

### Cuando un hilo está en I/O, Python permite que otro hilo siga trabajando.

### *Mientras un hilo espera, otro puede avanzar*

# Actividad
Vamos a implementar una descarga de dos URLs con la librería "requests" y su método "get(URL)". Primero haremos la descarga de forma secuencial, y después implementaremos una ejecución concurrente en hilos distintos.

## Descarga sin hilos (Secuencial)

In [4]:
# Actividad: Descarga secuencial

import requests
import time

def descargar(url):
    print("Descargando:", url)
    r = requests.get(url)
    print("Completado:", url, "| Tamaño:", len(r.text))

urls = ["https://siiau.udg.mx","https://www.youtube.com/"]

print("Descargando sin hilos")
inicio = time.time()

for u in urls:
    descargar(u)

fin = time.time()
print("Tiempo total:", fin - inicio)


Descargando sin hilos
Descargando: https://siiau.udg.mx
Completado: https://siiau.udg.mx | Tamaño: 20300
Descargando: https://www.youtube.com/
Completado: https://www.youtube.com/ | Tamaño: 681153
Tiempo total: 1.725569486618042


## Descarga usando hilos (Concurrencia)

In [5]:
# Actividad: Descarga concurrente con hilos

import threading
import requests
import time

def descargar(url):
    print("Descargando:", url)
    r = requests.get(url)
    print("Completado:", url, "| Tamaño:", len(r.text))

urls = ["https://siiau.udg.mx","https://www.youtube.com/"]

hilos = []

print("Descargando con hilos")
inicio = time.time()

for url in urls:
    t = threading.Thread(target=descargar, args=(url,))
    t.start()
    hilos.append(t)

for t in hilos:
    t.join()

fin = time.time()
print("Tiempo total:", fin - inicio)

Descargando con hilos
Descargando: https://siiau.udg.mx
Descargando: https://www.youtube.com/
Completado: https://www.youtube.com/ | Tamaño: 684908
Completado: https://siiau.udg.mx | Tamaño: 20300
Tiempo total: 0.42378807067871094


# Pensemos:
## 1. ¿Por qué la versión con hilos fue más rápida?
## 2. ¿Fue paralelismo real o concurrencia?
## 3. ¿Qué tipo de tareas en el campo de la programación y el Data Science podrían beneficiarse con este tipo de programación concurrente?
## 4. ¿Para qué tipo de tareas la concurrencia NO funcionaría bien?

# EJEMPLO DONDE LA PROGRAMACIÓN CONCURRENTE NO ES ÚTIL

In [6]:
# Ejemplo de tarea CPU-bound donde los hilos NO mejoran el rendimiento
# (por culpa del GIL)

import threading
import time

def calcular():
    total = 0
    for i in range(50_000_000):
        total += 1
    return total

inicio = time.time()
calcular()
fin = time.time()

print("Tiempo con ejecución SECUENCIAL:", fin - inicio, "segundos")

# Ahora intentamos con hilos

print("Ahora con hilos:")
inicio = time.time()

t1 = threading.Thread(target=calcular)
t2 = threading.Thread(target=calcular)

t1.start()
t2.start()

t1.join()
t2.join()

fin = time.time()

print("Tiempo con hilos:", fin - inicio, "segundos")
print("Observa que NO fue más rápido.")

Tiempo con ejecución SECUENCIAL: 2.691664695739746 segundos
Ahora con hilos:
Tiempo con hilos: 4.9932801723480225 segundos
Observa que NO fue más rápido.


## GIL (Global Interpreter Lock)
El GIL (Global Interpreter Lock) es un mecanismo interno de Python (CPython) que permite que solo un hilo ejecute código Python a la vez, incluso si tu computadora tiene varios núcleos, como un candado global.

Entonces:

• Solo un hilo puede ejecutar bytecode de Python en un momento dado.
• Los demás hilos deben esperar su turno.
• Python va “alternando” cuál hilo tiene el GIL.

Por lo tanto, si ejecutas dos hilos que hacen cálculos puros, no van a correr en paralelo. Además, se turnan y avanza primero uno un poquito, luego el otro, pero nunca juntos.

## ¿POR QUÉ EXISTE EL GIL?
Porque simplifica el diseño de Python:

* No hay que proteger cada estructura interna del intérprete con locks.
* Evita problemas complejos en la gestión de memoria.
* Hace que el intérprete sea más simple y más rápido para programas de un solo hilo.

Es una decisión de diseño histórica, no un error.

## ¿A QUIÉN AFECTA EL GIL?
Afecta solo a código Python puro, especialmente:

- loops grandes
- operaciones matemáticas
- procesamiento intensivo de datos
- algoritmos CPU-bound

## ¿CÓMO PODEMOS LIBRARNOS DEL GIL?
El GIL se libera automáticamente cuando un hilo hace operaciones de I/O:

- dormir (time.sleep())
- leer un archivo
- escribir un archivo
- hacer una petición HTTP
- acceder a una base de datos
- esperar una respuesta de red

Y también se libera cuando llamas a funciones optimizadas en C, como:

- NumPy
- Pandas
- OpenCV
- SciPy

# Conclusiones:

* La concurrencia organiza tareas para que no se bloqueen.
* El paralelismo requiere varios núcleos, pero la concurrencia no.
* Los hilos funcionan muy bien para tareas de I/O.
* No mejoran tareas de CPU intensiva.
* join() evita que el programa termine antes de que los hilos acaben.
* Python tiene un GIL que limita el paralelismo real en hilos.